# 4. Build the Agent Loop Yourself

An **agent** is an LLM that has access to tools and runs in a loop until it finishes the assigned task.

The LLM does the thinking and deciding. Your Python code runs the loop and controls which tools are actually allowed to run.

A simple agent loop looks like this:

```text
task -> LLM -> tool call -> Python runs tool -> result goes back to LLM -> done or keep going
```


## Today's Goal

Build one small agent that can check where the International Space Station is right now and turn that live data into a mission-control style update.

The task is finished when the model has enough information to give the final answer.


In [6]:
import json
import os
from pprint import pprint

import requests
# If you do not have python-dotenv installed, run: pip install python-dotenv
from dotenv import load_dotenv

TIMEOUT = 60
MODEL = "gpt-4.1-mini"

## Load the API Key

This is the same setup from notebook 3. The real key lives in `.env`.


In [7]:
load_dotenv()

api_key = os.environ["OPENAI_API_KEY"]

print("OPENAI_API_KEY loaded")

OPENAI_API_KEY loaded


In [8]:
openai_url = "https://api.openai.com/v1/responses"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

## Step 1: Write a Tool

A tool starts as a normal Python function.

This one calls two public APIs: one to track the International Space Station, and one to turn its latitude and longitude into a readable place name. No extra API key is needed.


In [9]:
def get_place_name(latitude, longitude):
    reverse_geocode_url = "https://api.bigdatacloud.net/data/reverse-geocode-client"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "localityLanguage": "en",
    }
    response = requests.get(reverse_geocode_url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    place_parts = [
        data.get("locality") or data.get("city"),
        data.get("principalSubdivision"),
        data.get("countryName"),
    ]
    place_name = ", ".join(part for part in place_parts if part)

    if place_name:
        return place_name

    return "an area with no nearby place name"


def get_iss_location():
    iss_url = "https://api.wheretheiss.at/v1/satellites/25544"
    response = requests.get(iss_url, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    latitude = data["latitude"]
    longitude = data["longitude"]

    return {
        "place_name": get_place_name(latitude, longitude),
        "latitude": round(latitude, 2),
        "longitude": round(longitude, 2),
        "altitude_km": round(data["altitude"], 2),
        "velocity_km_per_hour": round(data["velocity"], 2),
    }


In [11]:
pprint(get_iss_location())


{'altitude_km': 434.61,
 'latitude': -51.6,
 'longitude': -29.54,
 'place_name': 'Atlantic Ocean',
 'velocity_km_per_hour': 27546.36}


## Step 2: Describe the Tool to the Model

The model cannot see our Python function. It only sees this tool description.

This tool has no arguments, so `properties` and `required` are empty.


In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_iss_location",
        "description": (
            "Get the current place name, latitude, longitude, altitude, and speed of the "
            "International Space Station. Use this for live ISS tracking questions."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    }
]


## Step 3: Add Three Tiny Helpers

These helpers keep the loop readable:

- `ask_model()` sends one request to OpenAI
- `extract_text()` pulls out the final answer
- `run_tool()` runs only tools we approve


In [ ]:
def ask_model(input_value, previous_response_id=None, require_tool=False):
    payload = {
        "model": MODEL,
        "input": input_value,
        "tools": tools,
        "parallel_tool_calls": False,
        "max_output_tokens": 300,
    }

    if previous_response_id is not None:
        payload["previous_response_id"] = previous_response_id

    if require_tool:
        payload["tool_choice"] = "required"

    response = requests.post(
        openai_url,
        headers=headers,
        json=payload,
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def extract_text(response_data):
    pieces = []

    for item in response_data["output"]:
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                pieces.append(content["text"])

    return "\n".join(pieces)


def run_tool(function_call):
    if function_call["name"] == "get_iss_location":
        return get_iss_location()

    raise ValueError(f"Unknown tool: {function_call['name']}")


## Step 4: Build the Loop

Read this slowly. This is the whole agent pattern.

If the model returns text, we are done.

If the model returns a function call, Python runs the tool and sends the tool result back.


In [ ]:
def run_agent(question, max_rounds=3):
    response_data = ask_model(question, require_tool=True)

    for round_number in range(1, max_rounds + 1):
        function_calls = [
            item for item in response_data["output"]
            if item["type"] == "function_call"
        ]

        if not function_calls:
            print(f"Round {round_number}: final answer")
            return extract_text(response_data)

        print(f"Round {round_number}: model asked for a tool")

        tool_outputs = []
        for function_call in function_calls:
            print("Tool:", function_call["name"])

            tool_result = run_tool(function_call)
            pprint(tool_result)

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": function_call["call_id"],
                    "output": json.dumps(tool_result),
                }
            )

        response_data = ask_model(
            tool_outputs,
            previous_response_id=response_data["id"],
        )

    raise RuntimeError("The agent reached max_rounds before finishing.")


## Step 5: Launch It

Now we ask a question that needs live data.

The first round should be a tool call. The second round should be the final answer.


In [ ]:
answer = run_agent(
    "Give me a short mission-control update on the International Space Station. "
    "Use its current place name, coordinates, altitude, and speed. Make it exciting but factual."
)

print(answer)


## What Happened

Python and the model each had a job:

- The model decided it needed current ISS data.
- Python called the public ISS tracking API.
- Python used latitude and longitude to look up a readable place name.
- Python sent the tool result back to the model.
- The model wrote the final mission update.

That loop is what makes this an agent instead of a single prompt.


## Quick Review

- A **tool** is a Python function the model is allowed to ask for.
- A **function call** is the model saying, "Please run this tool."
- A **function call output** is Python sending the result back.
- An **agent loop** keeps going until the model gives a final answer.
- The model can choose the tool, but your code controls what actually runs.
